# Chapter 02: Refs, Branches & What a PR Really Is (Reference)

## Learning Objectives

- Explain what a Git ref is, independent of any GitHub-specific vocabulary
- Name the two refs GitHub maintains for every open PR and what each one means
- List PR refs with plain `git ls-remote` -- no GitHub token required
- Fetch a PR's higher-level object and read its `mergeable_state`
- State why `mergeable_state` can legitimately be `null`

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path` so `pr_automerge` is importable. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Reproducibility -- always set before any stochastic operation
RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

# PRA_ environment variables -- fixture mode by default so this notebook runs
# identically for every reader
PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. Layer 1: Raw Git Refs

The next cell calls `list_pr_refs` from `labs/lab_02_pr_refs.py`. In fixture mode this loads a canned ref listing; in live mode it shells out to `git ls-remote`. You should see a `refs/heads/main` entry plus two refs per open PR (`.../head` and `.../merge`).

In [ ]:
from labs.lab_02_pr_refs import list_pr_refs

refs = list_pr_refs("example/example")
for ref in refs:
    print(f"{ref['ref']:<28} {ref['sha'][:12]}")


## 2. Layer 2: The PR Object

The next cell calls `fetch_pr_object` and `summarize_pr_identity`. You should see `mergeable_state: 'clean'` in fixture mode -- contrast this field, which has no Git equivalent at all, against the raw refs printed above.

In [ ]:
from labs.lab_02_pr_refs import fetch_pr_object, summarize_pr_identity

pr = fetch_pr_object("example/example", 101)
identity = summarize_pr_identity(pr)
for key, value in identity.items():
    print(f"{key:<18}: {value}")


## 3. Confirming the Two Views Agree

The next cell checks that `refs/pull/101/head`'s SHA (Layer 1) and the PR object's `head_sha` (Layer 2) describe the *same* commit -- computed two completely different ways. You should see `MATCH`.

In [ ]:
head_ref = next(r for r in refs if r["ref"] == "refs/pull/101/head")
match = head_ref["sha"][:12] == identity["head_sha"]
print("MATCH" if match else "MISMATCH", "--", head_ref["sha"][:12], "vs", identity["head_sha"])


## Takeaways & Next Steps

This notebook's takeaway is the MATCH line above: the PR object isn't a separate source of truth from the refs -- it's a computed layer built on top of them.

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")


---

📖 **Reading companion:** [Chapter 02: Refs, Branches & What a PR Really Is](../learning_modules/chapter_02_refs_and_prs.md)
